# Optimizing Violation Finder
This notebook tests different patterns found in census and tax datasets to find the fastest way to detect violations.

In [ ]:
import pandas as pd
import numpy as np
import time
import duckdb
import sys
import os
sys.path.append(os.path.abspath('..'))

## Pattern 1: Conditional Constant-Value FD
`not(t1.AGEP=0 & t2.AGEP=0 & t1.SCHL=0 & t2.SCHL!=0)`
This means: If AGEP is 0, then SCHL must be 0. If we have a mix of SCHL=0 and SCHL!=0 for AGEP=0, they all conflict.

In [ ]:
n = 1000000
data = pd.DataFrame({
    'AGEP': np.random.choice([0, 1], size=n, p=[0.1, 0.9]),
    'SCHL': np.random.randint(0, 10, size=n)
})

def find_p1_pandas(df, limit=100000):
    start = time.time()
    # Filter for AGEP=0
    df_agep0 = df[df['AGEP'] == 0]
    if df_agep0.empty:
        return pd.DataFrame(), time.time() - start
        
    idx_schl0 = df_agep0[df_agep0['SCHL'] == 0].index
    idx_schl_not0 = df_agep0[df_agep0['SCHL'] != 0].index
    
    if len(idx_schl0) == 0 or len(idx_schl_not0) == 0:
        return pd.DataFrame(), time.time() - start
        
    # Cartesian product of idx_schl0 and idx_schl_not0
    # For performance, we limit the number of pairs created
    # This is a cross join of two ID lists
    res = []
    count = 0
    for i1 in idx_schl0:
        for i2 in idx_schl_not0:
            res.append((i1, i2))
            count += 1
            if count >= limit: break
        if count >= limit: break
    
    return pd.DataFrame(res, columns=['idx1', 'idx2']), time.time() - start

res_p1, t_p1 = find_p1_pandas(data)
print(f"Pandas P1 time: {t_p1:.4f}s, Found: {len(res_p1)}")

## Pattern 2: Standard FD (Single Key)
`not(t1.CIT=t2.CIT & t1.NATIVITY!=t2.NATIVITY)`

In [ ]:
data = pd.DataFrame({
    'CIT': np.random.randint(0, 50, size=n),
    'NATIVITY': np.random.randint(0, 2, size=n)
})

def find_p2_pandas(df, limit=100000):
    start = time.time()
    # Group by CIT and check if NATIVITY is unique
    groups = df.groupby('CIT')
    res = []
    count = 0
    
    for name, group in groups:
        if group['NATIVITY'].nunique() > 1:
            # Found a violating group. Now we need to find pairs with different NATIVITY
            vals = group['NATIVITY'].unique()
            # For each pair of different values, all cross pairs violate
            for i in range(len(vals)):
                for j in range(i+1, len(vals)):
                    idx1 = group[group['NATIVITY'] == vals[i]].index
                    idx2 = group[group['NATIVITY'] == vals[j]].index
                    for v1 in idx1:
                        for v2 in idx2:
                            res.append(tuple(sorted((v1, v2))))
                            count += 1
                            if count >= limit: break
                        if count >= limit: break
                    if count >= limit: break
                if count >= limit: break
        if count >= limit: break
            
    return pd.DataFrame(res, columns=['idx1', 'idx2']), time.time() - start

res_p2, t_p2 = find_p2_pandas(data)
print(f"Pandas P2 time: {t_p2:.4f}s, Found: {len(res_p2)}")